# 生成式 GPT 树 —— 用 logprobs 画出「下一个 token」分叉

## 练习目标（理念）

这是课程作者 **Ed Donner** 的 Colab 演示改写版：不只看模型最终答了什么，还用 **logprobs / top_logprobs** 观察每一步「差一点就选了谁」。

- **输入**：一段 prompt（本笔记本里默认是一条字体缺字形的 UserWarning 文本）
- **输出**：按 token 序列画出的有向图（主路径 + 备选 token）
- **额外**：同一 prompt 再走一遍普通 Chat Completions，用 Markdown 看完整回答

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| 流式 `stream=True` | 边生成边收集每个 token 的 logprobs |
| `logprobs` / `top_logprobs` | 每个位置上的候选 token 与概率 |
| `temperature` | 温度越高，越可能偏离最高概率 token |
| 可视化 | `networkx` 建图 + `matplotlib` 画树 |

## 怎么跑

1. 准备好 `.env`：`OPENAI_API_KEY`（或按原文在 Colab Secrets 里配置同名密钥）
2. 需要已安装：`openai`、`networkx`、`matplotlib`（见下一格可选 pip）
3. 从上到下运行；在「选模型并预测」那一格可改 `model_name` / `temperature` / `prompt`
4. 最后一格调用 `create_token_graph` + `visualize_predictions` 出图

### 密钥准备（原 Colab 说明的中文版）

1. 到 [OpenAI Platform](https://platform.openai.com/) 注册/登录 API 账号
2. 在 [Billing](https://platform.openai.com/settings/organization/billing/overview) 至少充值起步金额（常见最低约 $5）
3. 在 [API keys](https://platform.openai.com/settings/organization/api-keys) 创建密钥并妥善保存
4. 可到 [Playground](https://platform.openai.com/playground/chat?models=gpt-4o-mini) 确认账号可用

若在 Colab：左侧钥匙图标 → Name 填 `OPENAI_API_KEY` → 粘贴密钥 → 打开 Notebook access。本地则用 `.env` + `load_dotenv`。


In [ ]:
# ========== 依赖：OpenAI SDK + 图可视化库 ==========
# 首次环境可取消下一行注释安装；已在课程 venv 里则可跳过

#!pip install -q openai networkx


In [ ]:
# ========== 环境变量：从 .env / 环境读取 OPENAI_API_KEY ==========

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境
from dotenv import load_dotenv
# 导入标准库 os：用 getenv 读取环境变量
import os
# override=True：.env 覆盖已存在的同名环境变量
load_dotenv(override=True)
# 读出 API Key，供后面 OpenAI(api_key=...) 使用（变量名必须保持 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')


In [ ]:
# ========== 连通性冒烟测试：问一个极便宜的算术题 ==========

# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI
# 用上一格读到的 api_key 创建客户端（显式传入，而不是只靠默认环境）
openai = OpenAI(api_key=api_key)
# 一次最轻量的 Chat Completions：model 与 user 内容保持英文/原样
response = openai.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": "What is 2+2?"}])
# 打印模型回复文本，确认密钥与网络都正常
print(response.choices[0].message.content)


In [ ]:
# ========== 导入：建图、画图、类型标注、JSON、数学、Markdown 展示 ==========

# networkx：用有向图（DiGraph）表达 token 主链与备选分叉
import networkx as nx
# matplotlib.pyplot：把图节点/边画到图像上
import matplotlib.pyplot as plt
# typing：给预测列表等加 List / Dict / Tuple 类型标注，方便阅读
from typing import List, Dict, Tuple
# json：本格未强制使用，但常与结构化预测一起出现（保持原导入）
import json
# math：用 exp 把 log-probability 转成普通概率
import math
# IPython：后面用 Markdown 展示完整回答
from IPython.display import Markdown, display


In [ ]:
# ========== TokenPredictor：流式生成并记录每个 token 的 top_logprobs ==========

class TokenPredictor:
    """封装一次「按 token 流式生成 + 收集候选概率」的预测器。"""

    def __init__(self, client, model_name: str, temperature: int):
        # OpenAI 客户端实例（前面创建的 openai）
        self.client = client
        # 预留：多轮 messages（本练习 predict_tokens 直接传单条 user）
        self.messages = []
        # 预留：历史预测缓存（本练习主要用函数返回值）
        self.predictions = []
        # 模型 id 字符串，例如 gpt-4o（必须与账号可用模型一致）
        self.model_name = model_name
        # 采样温度：0 更贪心；调高会更常选非最高概率 token
        self.temperature = temperature

    def predict_tokens(self, prompt: str, max_tokens: int = 100) -> List[Dict]:
        """
        逐 token 流式生成，并跟踪每个位置的预测概率。
        返回列表：每项含选中 token、概率、以及若干备选 (token, prob)。
        """
        # stream=True：一边生成一边拿 chunk；logprobs + top_logprobs 打开候选分布
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=self.temperature,
            logprobs=True,
            # seed：尽量复现同分布采样（是否严格复现取决于服务端）
            seed=42,
            # 每个位置取 top-7 候选的 logprob
            top_logprobs=7,
            stream=True
        )

        predictions = []
        for chunk in response:
            # 仅处理带文本增量的 chunk（有的 chunk 只有 role / 结束标记）
            if chunk.choices[0].delta.content:
                # 本步实际发出的 token 文本
                token = chunk.choices[0].delta.content
                # 该位置 top_logprobs 列表（含 token 与 logprob）
                logprobs = chunk.choices[0].logprobs.content[0].top_logprobs
                # 建成 dict：token 字符串 → log 概率，便于查找
                logprob_dict = {item.token: item.logprob for item in logprobs}

                # 主路径：模型实际输出的 token 及其 logprob
                top_token = token
                top_prob = logprob_dict[token]

                # 备选：同一位置上其它高概率 token（排除已选中的）
                alternatives = []
                for alt_token, alt_prob in logprob_dict.items():
                    if alt_token != token:
                        # math.exp：log 概率 → 概率（0~1）
                        alternatives.append((alt_token, math.exp(alt_prob)))
                # 按概率从高到低排序备选
                alternatives.sort(key=lambda x: x[1], reverse=True)

                # 只保留前 2 个备选，控制图的宽度
                prediction = {'token': top_token, 'probability': math.exp(top_prob),'alternatives': alternatives[:2]}
                predictions.append(prediction)

        return predictions


In [ ]:
# ========== create_token_graph：把预测序列变成 networkx 有向图 ==========

def create_token_graph(model_name:str, predictions: List[Dict]) -> nx.DiGraph:
    """
    根据主 token 链与备选，创建有向图（DiGraph），供后面可视化。
    """
    # 空的有向图
    G = nx.DiGraph()

    # START 节点：用 model_name 标注，绿色大节点
    G.add_node("START", token=model_name, prob="START", color='lightgreen', size=4000)

    # 先按顺序创建主路径上的 token 节点 t0, t1, ...
    for i, pred in enumerate(predictions):
        token_id = f"t{i}"
        # 节点属性：显示 token 文本与百分比概率；蓝色主链
        G.add_node(token_id, token=pred['token'], prob=f"{pred['probability']*100:.1f}%", color='lightblue', size=6000)
        # 边：从前一个主节点（或 START）连到当前
        G.add_edge(f"t{i-1}" if i else "START", token_id)

    # 再为每个位置挂上备选节点（灰色），边从「父主节点」指出
    last_id = None
    for i, pred in enumerate(predictions):
        # 第 0 步的父是 START，否则是上一个主 token
        parent_token = "START" if i == 0 else f"t{i-1}"

        # 遍历该位置的备选 (alt_token, alt_prob)
        for j, (alt_token, alt_prob) in enumerate(pred['alternatives']):
            alt_id = f"t{i}_alt{j}"
            G.add_node(alt_id, token=alt_token, prob=f"{alt_prob*100:.1f}%", color='lightgray', size=6000)
            G.add_edge(parent_token, alt_id)

    # END 节点：挂在循环结束后的 parent_token 上（与原逻辑一致）
    G.add_node("END", token="END", prob="100%", color='red', size=6000)
    G.add_edge(parent_token, "END")
    return G


In [ ]:
# ========== visualize_predictions：用 Matplotlib 竖排画出 token 树 ==========

def visualize_predictions(G: nx.DiGraph, figsize=(14, 150)):
    """
    竖向布局可视化：主链居中，备选分列左右；figsize 很高以容纳长序列。
    """
    # 创建画布；默认高度很大，适合很多 token
    plt.figure(figsize=figsize)

    pos = {}
    # 主链节点垂直间距
    spacing_y = 10
    # 备选相对主链的水平偏移
    spacing_x = 5

    # 主节点：id 里不含 '_alt'（含 START / t* / END）
    main_nodes = [n for n in G.nodes() if '_alt' not in n]
    for i, node in enumerate(main_nodes):
        # x=0 居中，y 向下递减
        pos[node] = (0, -i * spacing_y)  # Center main tokens vertically

    # 备选节点：放在对应主 token 的左右两侧
    for node in G.nodes():
        if '_alt' in node:
            # 从 id 解析：t3_alt0 → 主 id t3、备选编号 0
            main_token = node.split('_')[0]
            alt_num = int(node.split('_alt')[1])
            if main_token in pos:
                # 第 0 个备选偏左，第 1 个偏右
                x_offset = spacing_x if alt_num else -spacing_x
                pos[node] = (x_offset, pos[main_token][1] + 0.05)

    # 画节点：颜色与大小来自图属性
    node_colors = [G.nodes[node]['color'] for node in G.nodes()]
    node_sizes = [G.nodes[node]['size'] for node in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes)

    # 画有向边（箭头）
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, alpha=0.7)

    # 标签：token 文本 + 概率（两行）
    labels = {node: f"{G.nodes[node]['token']}\n{G.nodes[node]['prob']}" for node in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels, font_size=14)

    # 标题字符串保持原样（作者调侃字体缺字形警告）
    plt.title("About the warning message..")
    # 关掉坐标轴，只看图结构
    plt.axis('off')

    # 按节点坐标加边距，避免裁切
    margin = 8
    x_values = [x for x, y in pos.values()]
    y_values = [y for x, y in pos.values()]
    plt.xlim(min(x_values) - margin, max(x_values) + margin)
    plt.ylim(min(y_values) - margin, max(y_values) + margin)
    # 在 Notebook 中显示图像
    plt.show()


## 开跑前的扩展资源（作者链接）

- 在 LinkedIn 上 [联系我](https://www.linkedin.com/in/eddonner/)
- 在 X 上 [关注我](https://x.com/edwarddonner)
- 在 YouTube 看 [其它视频](https://www.youtube.com/@Edward.Donner)
- 最重要的：考虑参加 8 周密集课 [Master LLM engineering](https://www.udemy.com/course/llm-engineering-master-ai-and-large-language-models/?referralCode=35EB41EBB11DD247CF54)


In [ ]:
# ========== 选模型 / 温度 / prompt，跑 TokenPredictor ==========

# 进阶实验：把 temperature 调高（例如 0.4）
# 温度更高 → 偶尔不选最高概率 token → 输出更多样，树的分叉也更「有戏」

# 模型 id 保持原样（需账号有权访问）
model_name = "gpt-4o"
# 温度为 0：更接近贪心解码，便于观察「主路径 vs 备选」
temperature = 0.0

# 用全局 openai 客户端构造预测器
predictor = TokenPredictor(openai, model_name, temperature)
# 备选 prompt（描述「蓝色」）保留注释，便于你改回实验
#prompt = "How would you describe the color blue to someone who has never been able to see, in no more than 3 sentences."
# 当前实际 prompt：一条 Matplotlib 缺字形（Glyph 9 / Tab）的 UserWarning 原文；保持可运行字符串原样
prompt=r"""g:\projects\llm_engineering\.venv\Lib\site-packages\IPython\core\pylabtools.py:170: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)"""
# 流式预测：得到每个位置的 token / 概率 / 备选
predictions = predictor.predict_tokens(prompt)
# 建图与可视化可先注释，等 predictions 就绪后再跑最后一格
#G = create_token_graph(model_name, predictions)
#visualize_predictions(G)


In [ ]:
# ========== 对照：普通 Chat Completions 看完整自然语言回答 ==========

# messages：system 定助手角色，user 复用上一格的 prompt（字符串保持英文原样）
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": prompt}
]

# 非流式一次拿完整回复；model / temperature 与预测实验对齐
response = openai.chat.completions.create(model="gpt-4o", messages=messages,temperature=temperature)
# 取出回复文本（变量名 salida 保持原样，勿重命名）
salida=response.choices[0].message.content
# 用 Markdown 在 Notebook 里漂亮展示
display(Markdown(salida))


In [ ]:
# ========== 出图：用上一格得到的 predictions 画 token 树 ==========

# 主链 + 备选 → DiGraph
G = create_token_graph(model_name, predictions)
# Matplotlib 竖排可视化（可能触发缺字形警告——正是 prompt 在讨论的那类问题）
visualize_predictions(G)
